Create function for agent

In [0]:
spark.sql("USE moviebuff.default")

import numpy as np
import pandas as pd
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

In [0]:
embeddings_pdf = spark.table("gold_movie_embeddings").toPandas()
embedding_matrix = np.array(embeddings_pdf["embedding"].tolist())

print(f"Loaded {len(embeddings_pdf)} movie embeddings, shape: {embedding_matrix.shape}")
print(embeddings_pdf.columns.tolist())  # confirm language_name is present

In [0]:
def get_query_embedding(text):
    resp = w.serving_endpoints.query(
        name="databricks-bge-large-en",
        input=[text]
    )
    return np.array(resp.data[0].embedding)

In [0]:
def attach_watch_info(recs, country):
    if recs.empty:
        recs["available_on"] = None
        return recs
    
    movie_ids = recs["movie_id"].tolist()
    ids_str = ", ".join(str(i) for i in movie_ids)
    
    watch_df = spark.sql(f"""
        SELECT movie_id, platform_clean FROM silver_watch_providers
        WHERE movie_id IN ({ids_str}) AND country = '{country}' group by movie_id,platform_clean
    """).toPandas()
    
    watch_grouped = watch_df.groupby("movie_id")["platform_clean"].apply(list).reset_index()
    watch_grouped.columns = ["movie_id", "available_on"]
    
    recs = recs.merge(watch_grouped, on="movie_id", how="left")
    recs["available_on"] = recs["available_on"].apply(lambda x: x if isinstance(x, list) else ["Not available"])
    return recs

In [0]:
def recommend_by_text(query_text, top_n=10):
    query_embedding = get_query_embedding(query_text)
    
    sims = np.dot(embedding_matrix, query_embedding) / (
        np.linalg.norm(embedding_matrix, axis=1) * np.linalg.norm(query_embedding)
    )
    
    top_indices = np.argsort(sims)[::-1][:top_n]
    
    results = embeddings_pdf.iloc[top_indices][["movie_id", "title"]].copy()
    results["score"] = sims[top_indices]
    return results

In [0]:
def recommend_similar_to_movie(movie_title, top_n=10):
    match = embeddings_pdf[embeddings_pdf["title"].str.lower() == movie_title.lower()]
    if match.empty:
        return pd.DataFrame(columns=["movie_id", "title", "score"])
    
    idx = match.index[0]
    query_embedding = embedding_matrix[idx]
    
    sims = np.dot(embedding_matrix, query_embedding) / (
        np.linalg.norm(embedding_matrix, axis=1) * np.linalg.norm(query_embedding)
    )
    sims[idx] = -1  # exclude itself
    
    top_indices = np.argsort(sims)[::-1][:top_n]
    results = embeddings_pdf.iloc[top_indices][["movie_id", "title"]].copy()
    results["score"] = sims[top_indices]
    return results

In [0]:
def recommend_by_genre_filtered(genre_name, query_text, country="IN", top_n=5, sort_by="similarity", language="English"):
    valid_ids = spark.sql(f"""
        SELECT DISTINCT movie_id FROM silver_movie_genres 
        WHERE genre_name = '{genre_name}'
    """).toPandas()["movie_id"].tolist()
    
    if not valid_ids:
        return pd.DataFrame(columns=["movie_id", "title", "score", "available_on"])
    
    # apply language filter, unless explicitly asked for "any"/"all"
    if language and language.lower() not in ("any", "all", "null", "none"):
        lang_ids = spark.sql(f"""
            SELECT DISTINCT movie_id FROM silver_dim_movies
            WHERE language_name = '{language}'
        """).toPandas()["movie_id"].tolist()
        valid_ids = [mid for mid in valid_ids if mid in set(lang_ids)]
    
    if not valid_ids:
        return pd.DataFrame(columns=["movie_id", "title", "score", "available_on"])
    
    ids_str = ", ".join(str(i) for i in valid_ids)
    
    if sort_by == "popularity":
        recs = spark.sql(f"""
            SELECT movie_id, title, popularity as score
            FROM silver_dim_movies
            WHERE movie_id IN ({ids_str})
            ORDER BY popularity DESC
            LIMIT {top_n}
        """).toPandas()
    
    elif sort_by == "rating":
        recs = spark.sql(f"""
            SELECT movie_id, title, vote_average as score
            FROM silver_dim_movies
            WHERE movie_id IN ({ids_str}) AND vote_count >= 100
            ORDER BY vote_average DESC
            LIMIT {top_n}
        """).toPandas()
    
    else:  # similarity
        mask = embeddings_pdf["movie_id"].isin(valid_ids)
        filtered_pdf = embeddings_pdf[mask].reset_index(drop=True)
        filtered_matrix = embedding_matrix[mask.values]
        
        if len(filtered_pdf) == 0:
            return pd.DataFrame(columns=["movie_id", "title", "score", "available_on"])
        
        query_embedding = get_query_embedding(query_text)
        sims = np.dot(filtered_matrix, query_embedding) / (
            np.linalg.norm(filtered_matrix, axis=1) * np.linalg.norm(query_embedding)
        )
        top_indices = np.argsort(sims)[::-1][:top_n]
        recs = filtered_pdf.iloc[top_indices][["movie_id", "title"]].copy()
        recs["score"] = sims[top_indices]
    
    return attach_watch_info(recs, country)

In [0]:
def recommend_with_watch_info(query_text, country="IN", top_n=10):
    recs = recommend_by_text(query_text, top_n=top_n)
    return attach_watch_info(recs, country)

In [0]:
# print(recommend_by_genre_filtered("Romance", "feel-good charming love story", country="IN", top_n=3, sort_by="similarity"))